# Qwen2.5-Coder 3B ← 14B Knowledge Distillation




In [1]:
!pip install -q trl transformers datasets accelerate bitsandbytes peft torch
!pip install -q -U huggingface_hub
# Note: skipping flash-attn — PyTorch's built-in SDPA is ~85-90% as fast and ships out of the box.

In [2]:
# ════════════════════════════════════════════════════════════
#  Qwen2.5-Coder 3B ← 14B  Knowledge Distillation
#  Platform : Google Colab A100 80GB
# ════════════════════════════════════════════════════════════

# ── 0. Silence warnings ─────────────────────────────────────
import os
import warnings
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"]   = "false"
warnings.filterwarnings("ignore")

import transformers
transformers.logging.set_verbosity_error()

In [3]:
# ── 1. Imports ───────────────────────────────────────────────
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl.experimental.distillation import DistillationConfig, DistillationTrainer
from huggingface_hub import login
from google.colab import userdata

print("✅ Imports done")

✅ Imports done


In [4]:
# ── 2. Hugging Face login (token from Colab secrets) ─────────
# In Colab: click the 🔑 icon in left sidebar, add a secret named HF_TOKEN
# with your token value, and toggle "Notebook access" on.
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("✅ Logged into HF Hub")

✅ Logged into HF Hub


In [5]:
# ── 3. Model names ───────────────────────────────────────────
TEACHER_MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct"
STUDENT_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
OUTPUT_DIR    = "qwen2.5-coder-3b-distilled-from-14b"
HUB_MODEL_ID  = "Harsha901/qwen2.5-coder-3b-distilled-from-14b"

MAX_LENGTH    = 1024  # caps activation memory; raise if your code samples need it
SEED          = 42

In [6]:
# ── 4. Dataset ───────────────────────────────────────────────
print("Loading dataset...")
dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")

def format_messages(example):
    instruction = example["instruction"]
    user_input  = example.get("input", "").strip()
    content     = f"{instruction}\n\nInput:\n{user_input}" if user_input else instruction
    # Include the gold output as the assistant turn — this gives the trainer
    # off-policy targets when lmbda < 1.0.
    return {
        "messages": [
            {"role": "user",      "content": content},
            {"role": "assistant", "content": example["output"]},
        ]
    }

dataset = dataset.map(
    format_messages,
    remove_columns=dataset.column_names,
)

split         = dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"✅ Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print(f"Sample: {train_dataset[0]}")

Loading dataset...
✅ Train: 17681 | Eval: 931
Sample: {'messages': [{'role': 'user', 'content': 'Create a Django application with two models: Post and Comment. Each Post should have a title, text, and a list of associated Comment objects.\n\nInput:\nNot applicable'}, {'role': 'assistant', 'content': "from django.db import models\n\n# Create your models here.\nclass Post(models.Model):\n title = models.CharField(max_length=255)\n text = models.TextField()\n\nclass Comment(models.Model):\n post = models.ForeignKey(Post, on_delete=models.CASCADE)\n text = models.TextField()\n\n# In your migrations file:\n\nfrom django.db import migrations, models\n\ndef create_posts_and_comments(apps, schema_editor):\n Post = apps.get_model('app', 'Post')\n Comment = apps.get_model('app', 'Comment')\n\n post = Post.objects.create(\n title='My first post',\n text='Hello world!'\n )\n\n Comment.objects.create(\n post=post,\n text='Nice post!'\n )\n\nclass Migration(migrations.Migration):\n dependencies = [\

In [7]:
# ── 5. Tokenizer ─────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Sanity check: GKD-style distillation requires identical tokenizers
teacher_tok = AutoTokenizer.from_pretrained(TEACHER_MODEL, trust_remote_code=True)
assert tokenizer.get_vocab() == teacher_tok.get_vocab(), \
    "Teacher and student vocabularies differ — distillation will produce garbage."
del teacher_tok

print(f"✅ Tokenizer loaded | Vocab size: {len(tokenizer)}")
print(f"   pad_token : {tokenizer.pad_token} ({tokenizer.pad_token_id})")
print(f"   eos_token : {tokenizer.eos_token} ({tokenizer.eos_token_id})")

Loading tokenizer...
✅ Tokenizer loaded | Vocab size: 151665
   pad_token : <|endoftext|> (151643)
   eos_token : <|im_end|> (151645)


In [8]:
# ── 6. QLoRA config (shared by both models) ──────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [9]:
# ── 7. Student model (trainable, QLoRA) ──────────────────────
print(f"\nLoading student: {STUDENT_MODEL} ...")
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="sdpa",
)

student_model = prepare_model_for_kbit_training(
    student_model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()
print(f"✅ Student loaded | device: {next(student_model.parameters()).device}")


Loading student: Qwen/Qwen2.5-Coder-3B-Instruct ...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
✅ Student loaded | device: cuda:0


In [10]:
# ── 8. Teacher model (frozen, inference only) ────────────────
print(f"\nLoading teacher: {TEACHER_MODEL} ...")
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="sdpa",
)

teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

print(f"✅ Teacher loaded and frozen | device: {next(teacher_model.parameters()).device}")

# ── VRAM check ──────────────────────────────────────────────
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n📊 VRAM | Allocated: {allocated:.1f}GB | Reserved: {reserved:.1f}GB | Total: {total:.1f}GB")


Loading teacher: Qwen/Qwen2.5-Coder-14B-Instruct ...


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

✅ Teacher loaded and frozen | device: cuda:0

📊 VRAM | Allocated: 12.8GB | Reserved: 33.3GB | Total: 85.1GB


In [11]:
# ── 9. Training plan ─────────────────────────────────────────
# Effective batch size = per_device_train_batch_size * gradient_accumulation_steps
#                      = 2 * 16 = 32   (same as previous run, but safer for VRAM)
#
# Previous run: 50 steps in 2 hours → ~2.4 min/step with lmbda=1.0.
# Dropping to lmbda=0.25 should roughly halve step time (less student generation).
#
# At ~1.2 min/step, 300 steps ≈ 6 hours, 500 steps ≈ 10 hours.
# Pick MAX_STEPS based on how long you can keep the Colab session alive.
MAX_STEPS = 300

print(f"Effective batch size : {2 * 16}")
print(f"Steps planned        : {MAX_STEPS}")
print(f"Examples seen        : {MAX_STEPS * 32} (~{MAX_STEPS * 32 / len(train_dataset):.2f} epochs)")

Effective batch size : 32
Steps planned        : 300
Examples seen        : 9600 (~0.54 epochs)


In [12]:
# ── 10. Trainer ──────────────────────────────────────────────
trainer = DistillationTrainer(
    model=student_model,
    teacher_model=teacher_model,
    args=DistillationConfig(
        output_dir=OUTPUT_DIR,
        seed=SEED,

        # Distillation
        lmbda=0.25,   # was 1.0; mostly off-policy, much faster per step
        beta=0.5,     # symmetric JSD between teacher and student logits
        max_length=MAX_LENGTH,

        # Training schedule
        max_steps=MAX_STEPS,
        per_device_train_batch_size=16,    # was 16 — too large with both models in VRAM
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,   # keeps effective batch size at 32
        gradient_checkpointing=True,


        # Optimizer
        learning_rate=1.5e-4,
        lr_scheduler_type="cosine",
        warmup_steps=30,
        weight_decay=0.01,
        optim="paged_adamw_8bit",

        # Precision
        bf16=True,

        # Logging
        logging_steps=5,
        logging_strategy="steps",
        disable_tqdm=False,
        log_level="info",

        # Eval & Save (steps must align for load_best_model_at_end)
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,



        # Hub
        push_to_hub=True,
        hub_model_id=HUB_MODEL_ID,
        hub_strategy="every_save",
        report_to="tensorboard",
    ),

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

# Final check — make sure the trainer rendered messages correctly
print("Trainer ready. First example as the trainer sees it:")
print(trainer.train_dataset[0])

max_steps is given, it will override any value given in num_train_epochs
You are resizing the embedding layer without providing a `pad_to_multiple_of` parameter. This means that the new embedding dimension will be 151936. This might induce some performance reduction as *Tensor Cores* will not be available. For more details about this, or help on choosing the correct value for resizing, refer to this guide: https://docs.nvidia.com/deeplearning/performance/dl-performance-matrix-multiplication/index.html#requirements-tc


Trainer ready. First example as the trainer sees it:
{'messages': [{'role': 'user', 'content': 'Create a Django application with two models: Post and Comment. Each Post should have a title, text, and a list of associated Comment objects.\n\nInput:\nNot applicable'}, {'role': 'assistant', 'content': "from django.db import models\n\n# Create your models here.\nclass Post(models.Model):\n title = models.CharField(max_length=255)\n text = models.TextField()\n\nclass Comment(models.Model):\n post = models.ForeignKey(Post, on_delete=models.CASCADE)\n text = models.TextField()\n\n# In your migrations file:\n\nfrom django.db import migrations, models\n\ndef create_posts_and_comments(apps, schema_editor):\n Post = apps.get_model('app', 'Post')\n Comment = apps.get_model('app', 'Comment')\n\n post = Post.objects.create(\n title='My first post',\n text='Hello world!'\n )\n\n Comment.objects.create(\n post=post,\n text='Nice post!'\n )\n\nclass Migration(migrations.Migration):\n dependencies = [\n

In [13]:
# ── 11. Train ────────────────────────────────────────────────
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
skipped Embedding(151936, 2048): 296.75M params
skipped: 296.75M params
***** Running training *****
  Num examples = 17,681
  Num Epochs = 1
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 2
  Total optimization steps = 300
  Number of trainable parameters = 29,933,568
Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss,Validation Loss
50,0.028746,0.012964
100,0.025177,0.012293
150,0.025299,0.011955
200,0.022998,0.011729
250,0.025547,0.011615
300,0.027076,0.011592



***** Running Evaluation *****
  Num examples = 931
  Batch size = 16
Saving model checkpoint to qwen2.5-coder-3b-distilled-from-14b/checkpoint-50
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-3B-Instruct/snapshots/488639f1ff808d1d3d0ba301aef8c11461451ec5/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_a

TrainOutput(global_step=300, training_loss=0.026540686438481013, metrics={'train_runtime': 18595.6116, 'train_samples_per_second': 0.516, 'train_steps_per_second': 0.016, 'total_flos': 2.2096211310123418e+17, 'train_loss': 0.026540686438481013})

In [14]:
# ── 12. Save and push to Hub ─────────────────────────────────
print("Saving model locally...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

print(f"\nPushing to Hub ({HUB_MODEL_ID})...")
trainer.push_to_hub()
print("✅ Successfully pushed to Hub")

Saving model checkpoint to qwen2.5-coder-3b-distilled-from-14b


Saving model locally...


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-3B-Instruct/snapshots/488639f1ff808d1d3d0ba301aef8c11461451ec5/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

  ...9136.d5fa247b696c.28844.0: 100%|##########| 59.8kB / 59.8kB            

  ...d-from-14b/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter_model.safetensors: 100%|##########|  120MB /  120MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Successfully pushed to Hub


In [19]:
"""
Merge LoRA adapter from distillation run into base model and push to HF Hub.

If you hit "torchao incompatible" error, run this in a separate cell first,
then restart the runtime (Runtime → Restart session):
    !pip install -q -U torchao

Usage in Colab:
    !python merge_and_push.py
"""

import os
import gc
import warnings
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# ─── Config ──────────────────────────────────────────────────────────
BASE_MODEL    = "Qwen/Qwen2.5-Coder-3B-Instruct"
ADAPTER_PATH  = "Harsha901/qwen2.5-coder-3b-distilled-from-14b"

MERGED_LOCAL_DIR = "qwen2.5-coder-3b-distilled-merged"
MERGED_HUB_REPO  = "Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged"


# ─── HF login ────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("✅ Logged in via Colab secrets")
except Exception:
    pass


# ─── Free VRAM ───────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"📊 VRAM at start: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


# ─── 1. Load base in bf16 (NOT 4-bit — merging quantized weights is lossy) ───
print(f"\nLoading base model in bf16: {BASE_MODEL}")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
if torch.cuda.is_available():
    print(f"📊 VRAM after base load: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


# ─── 2. Tokenizer ────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


# ─── 3. Attach LoRA adapter ──────────────────────────────────────────
print(f"\nAttaching LoRA adapter: {ADAPTER_PATH}")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print("✅ Adapter attached")


# ─── 4. Merge ────────────────────────────────────────────────────────
print("\nMerging adapter into base model...")
merged_model = model.merge_and_unload()
print("✅ Merge complete")
if torch.cuda.is_available():
    print(f"📊 VRAM after merge: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


# ─── 5. Save locally ─────────────────────────────────────────────────
print(f"\nSaving merged model locally to {MERGED_LOCAL_DIR}/")
merged_model.save_pretrained(
    MERGED_LOCAL_DIR,
    safe_serialization=True,
    max_shard_size="5GB",
)
tokenizer.save_pretrained(MERGED_LOCAL_DIR)
print(f"✅ Saved to {MERGED_LOCAL_DIR}/")


# ─── 6. Push to Hub (no extra kwargs — newer transformers dropped them) ──────
print(f"\nPushing to Hub: {MERGED_HUB_REPO}")
print("(2-6 min depending on connection — 3B bf16 is ~6 GB)")
merged_model.push_to_hub(MERGED_HUB_REPO)
tokenizer.push_to_hub(MERGED_HUB_REPO)

print(f"\n{'='*60}")
print(f"✅ DONE")
print(f"{'='*60}")
print(f"Merged model: https://huggingface.co/{MERGED_HUB_REPO}")
print(f"Local copy:   {MERGED_LOCAL_DIR}/")
print()
print("To load it later:")
print(f'  from transformers import AutoModelForCausalLM, AutoTokenizer')
print(f'  model = AutoModelForCausalLM.from_pretrained("{MERGED_HUB_REPO}")')
print(f'  tokenizer = AutoTokenizer.from_pretrained("{MERGED_HUB_REPO}")')

✅ Logged in via Colab secrets
📊 VRAM at start: 45.1 GB

Loading base model in bf16: Qwen/Qwen2.5-Coder-3B-Instruct


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-3B-Instruct/snapshots/488639f1ff808d1d3d0ba301aef8c11461451ec5/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-3B-Instruct/snapshots/488639f1ff808d1d3d0ba301aef8c11461451ec5/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

Could not locate the custom_generate/generate.py inside Qwen/Qwen2.5-Coder-3B-Instruct.


📊 VRAM after base load: 51.2 GB


...

chat template saved in /tmp/tmp2th9kgxn/chat_template.jinja
tokenizer config file saved in /tmp/tmp2th9kgxn/tokenizer_config.json
Uploading the following files to Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged: tokenizer.json,chat_template.jinja,README.md,tokenizer_config.json


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp2th9kgxn/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            


✅ DONE
Merged model: https://huggingface.co/Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged
Local copy:   qwen2.5-coder-3b-distilled-merged/

To load it later:
  from transformers import AutoModelForCausalLM, AutoTokenizer
  model = AutoModelForCausalLM.from_pretrained("Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged")
  tokenizer = AutoTokenizer.from_pretrained("Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged")


In [23]:
"""
Eval 1/2: Loss comparison on Magicoder training data.

Compares base 3B vs your merged distilled 3B on the training dataset.

What this tells you:
  - If merged loss << base loss → distillation fit the training distribution (good)
  - If merged loss ≈ base loss → training had no effect (or merge silently broke)
  - If merged loss >> base loss → something is seriously wrong

This is a sanity check, not a capability test. For capability, use eval_humaneval.py.

Runtime: ~15-20 min on A100 for 500 examples.
"""

import os
import gc
import warnings
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# ─── Config ──────────────────────────────────────────────────────────
BASE_MODEL       = "Qwen/Qwen2.5-Coder-3B-Instruct"
MERGED_MODEL     = "Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged"

DATASET          = "ise-uiuc/Magicoder-OSS-Instruct-75K-Instruction-Response"
NUM_EVAL_SAMPLES = 10000
MAX_LENGTH       = 1536
SEED             = 42
BATCH_SIZE       = 4

# ─── HF login ────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    pass


# ─── Tokenizer ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"


# ─── Dataset (same split seed as training) ───────────────────────────
print(f"Loading {DATASET}...")
ds = load_dataset(DATASET, split="train")

def format_messages(example):
    return {
        "messages": [
            {"role": "user",      "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]},
        ]
    }

ds = ds.map(format_messages, remove_columns=ds.column_names)
split = ds.train_test_split(test_size=0.05, seed=SEED)
eval_ds = split["test"].select(range(min(NUM_EVAL_SAMPLES, len(split["test"]))))
print(f"Eval examples: {len(eval_ds)}")


def encode_example(example):
    msgs = example["messages"]
    prompt_only = tokenizer.apply_chat_template(
        msgs[:-1], tokenize=False, add_generation_prompt=True,
    )
    full = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False,
    )
    full_ids = tokenizer(full, truncation=True, max_length=MAX_LENGTH)["input_ids"]
    prompt_ids = tokenizer(prompt_only, truncation=True, max_length=MAX_LENGTH)["input_ids"]
    labels = list(full_ids)
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100
    return {
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * len(full_ids),
    }

print("Tokenizing...")
eval_encoded = eval_ds.map(encode_example, remove_columns=eval_ds.column_names)


def collate(batch):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, attention_mask = [], [], []
    for b in batch:
        pad_n = max_len - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [tokenizer.pad_token_id] * pad_n)
        labels.append(b["labels"] + [-100] * pad_n)
        attention_mask.append(b["attention_mask"] + [0] * pad_n)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attention_mask),
    }


@torch.no_grad()
def compute_loss(model, eval_data, label):
    model.eval()
    losses, n_tokens = [], []
    for i in tqdm(range(0, len(eval_data), BATCH_SIZE), desc=label):
        batch_raw = [eval_data[j] for j in range(i, min(i + BATCH_SIZE, len(eval_data)))]
        batch = collate(batch_raw)
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits[:, :-1, :].contiguous()
        labels = batch["labels"][:, 1:].contiguous()
        loss_fn = torch.nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
        flat = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1)).view(labels.size())
        mask = (labels != -100).float()
        per_ex = (flat * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        per_tok = mask.sum(dim=1)
        losses.extend(per_ex.cpu().tolist())
        n_tokens.extend(per_tok.cpu().tolist())
    return np.array(losses), np.array(n_tokens)


# ─── 4-bit (fits in low VRAM; same quant for both = fair) ────────────
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


# ─── Eval base ───────────────────────────────────────────────────────
print(f"\n{'='*60}\nLoading BASE: {BASE_MODEL}\n{'='*60}")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="sdpa",
)
base_losses, base_tokens = compute_loss(base, eval_encoded, "Base 3B")
del base
gc.collect()
torch.cuda.empty_cache()


# ─── Eval merged ─────────────────────────────────────────────────────
print(f"\n{'='*60}\nLoading MERGED: {MERGED_MODEL}\n{'='*60}")
merged = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="sdpa",
)
merged_losses, merged_tokens = compute_loss(merged, eval_encoded, "Merged 3B")
del merged
gc.collect()
torch.cuda.empty_cache()


# ─── Report ──────────────────────────────────────────────────────────
def stats(losses, tokens):
    return {
        "weighted": (losses * tokens).sum() / tokens.sum(),
        "mean":     losses.mean(),
        "median":   np.median(losses),
        "p25":      np.percentile(losses, 25),
        "p75":      np.percentile(losses, 75),
    }

b = stats(base_losses, base_tokens)
m = stats(merged_losses, merged_tokens)

print(f"\n{'='*70}")
print(f"LOSS COMPARISON ({len(eval_ds)} examples on {DATASET})")
print(f"{'='*70}")
print(f"{'Metric':<15} {'Base 3B':>15} {'Merged 3B':>15} {'Δ':>12} {'Δ %':>10}")
print("-" * 70)
for k in ["weighted", "mean", "median", "p25", "p75"]:
    delta = m[k] - b[k]
    delta_pct = (delta / b[k]) * 100
    print(f"{k:<15} {b[k]:>15.4f} {m[k]:>15.4f} {delta:>+8.4f}    {delta_pct:>+6.1f}%")

wins = (merged_losses < base_losses).sum()
print(f"\nMerged wins on {wins}/{len(merged_losses)} examples ({100*wins/len(merged_losses):.1f}%)")

delta_pct = (m["weighted"] - b["weighted"]) / b["weighted"] * 100
print(f"\n{'─'*70}")
if delta_pct < -10:
    print(f"✅ Strong fit: merged is {-delta_pct:.1f}% lower loss on training data")
elif delta_pct < -3:
    print(f"✓ Real fit: merged is {-delta_pct:.1f}% lower loss on training data")
elif delta_pct < 0:
    print(f"⚠️  Marginal ({-delta_pct:.1f}%) — training had minimal effect")
elif delta_pct < 1:
    print(f"❌ No improvement — adapter merged but didn't change behavior")
else:
    print(f"❌ Merged is WORSE by {delta_pct:.1f}% — merge likely broken")
print(f"{'─'*70}")
print(f"\nNext step: run eval_humaneval.py to test if this translates to real capability gains.")

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-3B-Instruct/snapshots/488639f1ff808d1d3d0ba301aef8c11461451ec5/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "

Loading ise-uiuc/Magicoder-OSS-Instruct-75K-Instruction-Response...
Eval examples: 3760
Tokenizing...


Map:   0%|          | 0/3760 [00:00<?, ? examples/s]


Loading BASE: Qwen/Qwen2.5-Coder-3B-Instruct


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-3B-Instruct/snapshots/488639f1ff808d1d3d0ba301aef8c11461451ec5/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "

...


Loading MERGED: Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Harsha901--qwen2.5-coder-3b-distilled-from-14b-merged/snapshots/08b5e323f4a635c023c43599cb9ee45f949e2b2a/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Harsha901--qwen2.5-coder-3b-distilled-from-14b-merged/snapshots/08b5e323f4a635c023c43599cb9ee45f949e2b2a/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

Could not locate the custom_generate/generate.py inside Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged.
Merged 3B: 100%|██████████| 940/940 [02:23<00:00,  6.54it/s]



LOSS COMPARISON (3760 examples on ise-uiuc/Magicoder-OSS-Instruct-75K-Instruction-Response)
Metric                  Base 3B       Merged 3B            Δ        Δ %
----------------------------------------------------------------------
weighted                 0.3434          0.3405  -0.0029      -0.9%
mean                     0.3468          0.3439  -0.0029      -0.8%
median                   0.3259          0.3208  -0.0050      -1.5%
p25                      0.2304          0.2285  -0.0019      -0.8%
p75                      0.4291          0.4279  -0.0012      -0.3%

Merged wins on 2171/3760 examples (57.7%)

──────────────────────────────────────────────────────────────────────
⚠️  Marginal (0.9%) — training had minimal effect
──────────────────────────────────────────────────────────────────────

Next step: run eval_humaneval.py to test if this translates to real capability gains.


In [24]:
"""
Eval 2/2: HumanEval pass@1 — functional correctness on Python coding tasks.

Generates code completions from both base and merged 3B models, executes the
generated code against the provided unit tests, and reports pass rates.

This is the test that actually matters: does your distilled model write code
that runs correctly more often than the base?

HumanEval has 164 problems. Greedy decoding, one sample per problem.
Runtime: ~30-45 min total on A100 for both models.

⚠️  WARNING: This executes generated code. Run in a Colab/sandboxed environment,
    not on a machine with sensitive data. The code-execution sandbox in this
    script uses subprocess with timeout but is not a true security boundary.
"""

import os
import gc
import re
import warnings
import subprocess
import tempfile
import signal
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# ─── Config ──────────────────────────────────────────────────────────
BASE_MODEL   = "Qwen/Qwen2.5-Coder-3B-Instruct"
MERGED_MODEL = "Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged"

MAX_NEW_TOKENS = 512
EXEC_TIMEOUT   = 10        # seconds per problem
BATCH_GEN      = False     # batch generation is complex with chat templates; do one at a time

# ─── HF login ────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    pass


# ─── Load HumanEval ──────────────────────────────────────────────────
print("Loading HumanEval...")
he = load_dataset("openai/openai_humaneval", split="test")
print(f"Problems: {len(he)}")
print(f"Sample fields: {list(he[0].keys())}")


# ─── Prompt formatter ────────────────────────────────────────────────
def build_prompt(problem):
    """Wrap HumanEval prompt in an instruction-style request."""
    return (
        f"Complete the following Python function. Return only the completed code "
        f"in a code block, no explanation:\n\n```python\n{problem['prompt']}```"
    )


# ─── Extract code from model output ──────────────────────────────────
def extract_code(text, original_prompt):
    """Pull Python code out of the model's response.

    HumanEval grading needs `prompt + completion` to form a valid program.
    The model may return the full function (including the signature from prompt)
    or just the body. We handle both.
    """
    # Try to find a ```python ... ``` block first
    code_block = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if code_block:
        code = code_block.group(1)
    else:
        # No code fence — assume the whole response is code
        code = text

    # If the model returned the full function (including signature from prompt),
    # use it as-is. Otherwise prepend the prompt.
    # Check by seeing if `def <func_name>` appears in the extracted code.
    func_name_match = re.search(r"def\s+(\w+)\s*\(", original_prompt)
    if func_name_match:
        func_name = func_name_match.group(1)
        if f"def {func_name}" in code:
            return code  # full function returned
    # Otherwise treat as body-only and concat
    return original_prompt + code


# ─── Code execution with timeout ─────────────────────────────────────
def run_test(full_code, test_code, entry_point, timeout=EXEC_TIMEOUT):
    """Run the candidate code + test in a subprocess. Returns True if passes."""
    script = (
        full_code
        + "\n\n"
        + test_code
        + f"\n\ncheck({entry_point})\n"
    )

    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(script)
        path = f.name

    try:
        result = subprocess.run(
            ["python3", path],
            capture_output=True,
            timeout=timeout,
        )
        return result.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    except Exception:
        return False
    finally:
        try:
            os.unlink(path)
        except Exception:
            pass


# ─── Generate + grade for one model ──────────────────────────────────
def evaluate_model(model_name, tokenizer):
    print(f"\n{'='*60}\nLoading {model_name}\n{'='*60}")

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    model.eval()
    if torch.cuda.is_available():
        print(f"📊 VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    results = []
    for problem in tqdm(he, desc=f"Eval {model_name.split('/')[-1]}"):
        prompt = build_prompt(problem)
        msgs = [{"role": "user", "content": prompt}]
        encoded = tokenizer.apply_chat_template(
            msgs,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        input_ids = encoded["input_ids"].to(model.device)
        attention_mask = encoded["attention_mask"].to(model.device)
        prompt_len = input_ids.shape[1]

        with torch.no_grad():
            out = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        generated = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

        full_code = extract_code(generated, problem["prompt"])
        passed = run_test(
            full_code,
            problem["test"],
            problem["entry_point"],
        )
        results.append({
            "task_id":  problem["task_id"],
            "passed":   passed,
            "generated": generated,
        })

    del model
    gc.collect()
    torch.cuda.empty_cache()

    n_pass = sum(r["passed"] for r in results)
    return results, n_pass


# ─── Run both models ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"\n{'#'*70}\n# Evaluating BASE\n{'#'*70}")
base_results, base_pass = evaluate_model(BASE_MODEL, tokenizer)

print(f"\n{'#'*70}\n# Evaluating MERGED\n{'#'*70}")
merged_results, merged_pass = evaluate_model(MERGED_MODEL, tokenizer)


# ─── Report ──────────────────────────────────────────────────────────
n = len(he)
base_pct = 100 * base_pass / n
merged_pct = 100 * merged_pass / n
delta_pp = merged_pct - base_pct

print(f"\n{'='*70}")
print(f"HUMANEVAL PASS@1 RESULTS")
print(f"{'='*70}")
print(f"{'Model':<20} {'Passed':>10} {'Total':>8} {'Pass@1':>10}")
print("-" * 50)
print(f"{'Base 3B':<20} {base_pass:>10} {n:>8} {base_pct:>9.1f}%")
print(f"{'Merged 3B':<20} {merged_pass:>10} {n:>8} {merged_pct:>9.1f}%")
print(f"{'Δ':<20} {merged_pass-base_pass:>+10} {'':>8} {delta_pp:>+9.1f}pp")

# Per-problem breakdown
base_pass_ids   = {r["task_id"] for r in base_results if r["passed"]}
merged_pass_ids = {r["task_id"] for r in merged_results if r["passed"]}

both    = base_pass_ids & merged_pass_ids
only_b  = base_pass_ids - merged_pass_ids
only_m  = merged_pass_ids - base_pass_ids
neither = n - len(base_pass_ids | merged_pass_ids)

print(f"\nPer-problem breakdown:")
print(f"  Both pass        : {len(both):>4}")
print(f"  Only base passes : {len(only_b):>4}  (distillation regressed these)")
print(f"  Only merged pass : {len(only_m):>4}  (distillation gained these)")
print(f"  Neither passes   : {neither:>4}")
print(f"  Net change       : {len(only_m) - len(only_b):>+4} problems")

print(f"\n{'─'*70}")
if delta_pp > 3:
    print(f"✅ Real improvement: +{delta_pp:.1f}pp on HumanEval. Distillation worked.")
elif delta_pp > 1:
    print(f"✓ Modest improvement: +{delta_pp:.1f}pp. Possibly real, possibly noise.")
elif delta_pp > -1:
    print(f"⚠️  No meaningful change ({delta_pp:+.1f}pp). Distillation didn't help on HumanEval.")
else:
    print(f"❌ Regression of {-delta_pp:.1f}pp. Distillation made the model worse.")
print(f"{'─'*70}")
print()
print("Note: HumanEval pass@1 with greedy decoding has ~±2pp noise. Differences")
print("smaller than 3pp may not be statistically meaningful.")

# Save full results for inspection
import json
with open("humaneval_results.json", "w") as f:
    json.dump({
        "base":   base_results,
        "merged": merged_results,
        "summary": {
            "base_pass":   base_pass,
            "merged_pass": merged_pass,
            "total":       n,
            "base_pct":    base_pct,
            "merged_pct":  merged_pct,
            "delta_pp":    delta_pp,
        }
    }, f, indent=2)
print(f"\nFull results saved to humaneval_results.json")

Loading HumanEval...


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Problems: 164
Sample fields: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point']


...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Harsha901--qwen2.5-coder-3b-distilled-from-14b-merged/snapshots/08b5e323f4a635c023c43599cb9ee45f949e2b2a/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

Could not locate the custom_generate/generate.py inside Harsha901/qwen2.5-coder-3b-distilled-from-14b-merged.


📊 VRAM: 22.6 GB


Eval qwen2.5-coder-3b-distilled-from-14b-merged: 100%|██████████| 164/164 [32:14<00:00, 11.80s/it]



HUMANEVAL PASS@1 RESULTS
Model                    Passed    Total     Pass@1
--------------------------------------------------
Base 3B                     133      164      81.1%
Merged 3B                   137      164      83.5%
Δ                            +4               +2.4pp

Per-problem breakdown:
  Both pass        :  126
  Only base passes :    7  (distillation regressed these)
  Only merged pass :   11  (distillation gained these)
  Neither passes   :   20
  Net change       :   +4 problems

──────────────────────────────────────────────────────────────────────
✓ Modest improvement: +2.4pp. Possibly real, possibly noise.
──────────────────────────────────────────────────────────────────────

Note: HumanEval pass@1 with greedy decoding has ~±2pp noise. Differences
smaller than 3pp may not be statistically meaningful.

Full results saved to humaneval_results.json
